# M.Tech Thesis — Cross-lingual Hindi→Bhojpuri Dependency Parser

**Systems:**
- **System F** — High-quality fine-tuning on professor's Bhojpuri data
- **System G** — Exact alignment joint training (Hindi + Bhojpuri MSE)
- **System H** — Syntax-Aware Cross-lingual Transfer (SACT) ← novel contribution

**Before running:**
1. Upload `bhojpuri_matched_transferred.conllu` and `hindi_matched.conllu` to a folder called `thesis_data` in your Google Drive root.
2. Set Runtime → Change runtime type → **GPU (T4)**.
3. Run cells **in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14**.

**On Kaggle:** use T4 GPU (not P100). In Cell 5 replace the Drive mount with your Kaggle dataset copy.

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > GPU')

## Cell 2 — Install dependencies
**Do NOT downgrade numpy** — Colab's compiled extensions require numpy 2.x.

In [ ]:
%%capture
!pip install "torch==2.2.2" --index-url https://download.pytorch.org/whl/cu118
!pip install "transformers==4.36.2" tokenizers sentencepiece
!pip install trankit
# Keep Colab's numpy 2.x — do NOT downgrade

## Cell 3 — Patch trankit for Python 3.12 + numpy 2.x
Trankit was written for Python 3.9 and numpy 1.x. This cell patches the installed source.

In [ ]:
import re, pathlib, sys

# ── Fix 1: Python 3.12 disallows mutable defaults in dataclasses ─────────────
f = pathlib.Path('/usr/local/lib/python3.12/dist-packages/trankit/adapter_transformers/adapter_config.py')
txt = f.read_text()
if 'from dataclasses import dataclass, field' not in txt:
    txt = txt.replace('from dataclasses import dataclass',
                      'from dataclasses import dataclass, field')
def fix_mutable(m):
    return f'{m.group(1)} = field(default_factory={m.group(2)})'
txt = re.sub(r'( {4}\w+:\s*\w+Config)\s*=\s*(\w+Config)\(\)', fix_mutable, txt)
f.write_text(txt)
print('Patch 1 OK: adapter_config.py mutable defaults fixed')

# ── Fix 2: numpy 2.0 removed np.bool / np.int / np.float aliases ─────────────
for p in pathlib.Path('/usr/local/lib/python3.12/dist-packages/trankit').rglob('*.py'):
    src = p.read_text()
    changed = src
    changed = re.sub(r'\bnp\.bool\b(?!_)',    'np.bool_',     changed)
    changed = re.sub(r'\bnp\.int\b(?!_)',     'np.int_',      changed)
    changed = re.sub(r'\bnp\.float\b(?!_)',   'np.float64',   changed)
    changed = re.sub(r'\bnp\.complex\b(?!_)', 'np.complex128',changed)
    if changed != src:
        p.write_text(changed)
        print(f'  Patch 2 OK: {p.name}')

# ── Verify ────────────────────────────────────────────────────────────────────
for mod in list(sys.modules):
    if 'trankit' in mod: del sys.modules[mod]

from trankit import TPipeline
print('\ntrankit import OK — ready to train')

## Cell 4 — Download XLM-RoBERTa (pytorch_model.bin for trankit)

In [ ]:
# Cell 4 — Download XLM-RoBERTa (pytorch_model.bin format for trankit)
import os, shutil, pathlib, site

# Platform-aware local path (works on Kaggle and Colab)
LOCAL_XLM = (
    '/kaggle/working/xlm-roberta-base' if os.path.isdir('/kaggle/working')
    else '/content/xlm-roberta-base'
)

# ── Step 1: Remove any stale partial download (no pytorch_model.bin) ─────────
if pathlib.Path(LOCAL_XLM).is_dir() and not (pathlib.Path(LOCAL_XLM) / 'pytorch_model.bin').exists():
    print(f'Removing stale directory: {LOCAL_XLM}')
    shutil.rmtree(LOCAL_XLM)

# ── Step 2: Download (skip if already present) ────────────────────────────────
if not (pathlib.Path(LOCAL_XLM) / 'pytorch_model.bin').exists():
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    os.environ['HF_HUB_OFFLINE'] = '0'
    _cwd = os.getcwd()
    os.chdir('/tmp')  # prevent HF from resolving 'xlm-roberta-base' as a local dir
    from transformers import XLMRobertaModel, XLMRobertaTokenizerFast
    print('Downloading XLM-RoBERTa model weights (~1.1 GB)...')
    XLMRobertaModel.from_pretrained('xlm-roberta-base').save_pretrained(
        LOCAL_XLM, safe_serialization=False)
    print('Downloading tokenizer...')
    XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base').save_pretrained(LOCAL_XLM)
    os.chdir(_cwd)
    print('Download complete:', sorted(os.listdir(LOCAL_XLM)))
else:
    print('XLM-R already present:', LOCAL_XLM)

# ── Step 3: Export path so training scripts pick it up via env var ────────────
os.environ['XLM_R_PATH'] = LOCAL_XLM
print(f'XLM_R_PATH = {LOCAL_XLM}')

# ── Step 4: Patch trankit base_models.py to load from XLM_R_PATH ─────────────
for sp in site.getsitepackages():
    bm = pathlib.Path(sp) / 'trankit/models/base_models.py'
    if bm.exists():
        txt = bm.read_text()
        if 'XLM_R_PATH' not in txt:
            if 'import os' not in txt:
                txt = 'import os\n' + txt
            txt = txt.replace(
                'self.xlmr = XLMRobertaModel.from_pretrained(config.embedding_name,',
                'self.xlmr = XLMRobertaModel.from_pretrained(os.environ.get("XLM_R_PATH", config.embedding_name),'
            )
            bm.write_text(txt)
            print(f'Patched trankit base_models.py: {bm}')
        else:
            print('base_models.py already patched.')
        break

# ── Step 5: Patch tpipeline.py assertion to accept local paths ───────────────
for sp in site.getsitepackages():
    tp = pathlib.Path(sp) / 'trankit/tpipeline.py'
    if tp.exists():
        txt = tp.read_text()
        if 'isdir' not in txt and 'supported_embeddings' in txt:
            txt = txt.replace(
                'assert master_config.embedding_name in supported_embeddings,',
                'assert master_config.embedding_name in supported_embeddings or '
                '__import__("os").path.isdir(master_config.embedding_name),'
            )
            tp.write_text(txt)
            print(f'Patched tpipeline.py: {tp}')
        else:
            print('tpipeline.py already patched.')
        break


## Cell 5 — Clone the repository

In [ ]:
import os

REPO = 'https://github.com/Sansgithub22/mtechthesis4biaffinemultilingual-parser.git'
WORK = '/content/mtechthesis4biaffinemultilingual-parser'

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !cd {WORK} && git pull

os.chdir(WORK)
print('Working dir:', os.getcwd())

## Cell 6 — Mount Google Drive and copy data files

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, os

# ── EDIT THIS if your folder name in Drive is different ──
DRIVE_DATA = '/content/drive/MyDrive/thesis_data'

# Show what's in the folder so you can verify the filenames
print('Files in Drive thesis_data folder:')
try:
    print(os.listdir(DRIVE_DATA))
except FileNotFoundError:
    print('  ERROR: thesis_data folder not found in Google Drive!')
    print('  Create it at drive.google.com and upload the two .conllu files.')

os.makedirs('/content/mtechthesis4biaffinemultilingual-parser', exist_ok=True)

for fname in ['bhojpuri_matched_transferred.conllu', 'hindi_matched.conllu']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'/content/mtechthesis4biaffinemultilingual-parser/{fname}'
    if not os.path.exists(src):
        print(f'NOT FOUND: {src}')
    elif not os.path.exists(dst):
        print(f'Copying {fname} ...')
        shutil.copy(src, dst)
        print(f'  Done — {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'Already present: {fname} ({os.path.getsize(dst)/1e6:.1f} MB)')

## Cell 7 — Download UD data (Hindi treebank + Bhojpuri BHTB test)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
os.environ['TRANSFORMERS_OFFLINE'] = '0'
os.environ['HF_HUB_OFFLINE'] = '0'

!mkdir -p data_files/hindi data_files/bhojpuri data_files/sysf

HINDI_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master'
for split in ['train', 'dev', 'test']:
    dst = f'data_files/hindi/hi_hdtb-ud-{split}.conllu'
    if not os.path.exists(dst):
        !wget -q {HINDI_BASE}/hi_hdtb-ud-{split}.conllu -O {dst}
        print(f'Downloaded: {dst}')
    else:
        print(f'Already present: {dst}')

BHO_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Bhojpuri-BHTB/master'
dst = 'data_files/bhojpuri/bho_bhtb-ud-test.conllu'
if not os.path.exists(dst):
    !wget -q {BHO_BASE}/bho_bhtb-ud-test.conllu -O {dst}
    print(f'Downloaded: {dst}')
else:
    print(f'Already present: {dst}')

print('\nData files ready.')

## Cell 8 — Train Hindi model (System A) — warm-start source for F/G/H
~3-4 hours on T4. Saves checkpoint to `checkpoints/trankit_hindi/`.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_trankit_hindi.py --epochs 60 --batch_size 32 --gpu

## Cell 9 — Train System F (high-quality fine-tuning)
~2-3 hours on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_f.py --epochs 20 --batch_size 64 --gpu

## Cell 10 — Train System G (exact alignment joint training)
~2-3 hours on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_g.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_align 0.5

## Cell 11 — Train System H (Syntax-Aware Cross-lingual Transfer) ← Novel contribution
~2-3 hours on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_h.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_cosine 0.4 \
    --lambda_arc 0.1 \
    --lambda_cts 0.6 \
    --warmup_epochs 3

## Cell 12 — Train System I (Language-Agnostic SACT) ← Best novel system
~2-3 hours on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 train_system_i.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_cosine 0.4 \
    --lambda_arc 0.1 \
    --lambda_cts 0.6 \
    --lambda_adv 0.1 \
    --grl_alpha 1.0 \
    --warmup_epochs 3


## Cell 13 — Evaluate all systems on BHTB test set

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!python3 evaluate_trankit.py --gpu

## Cell 14 — Save checkpoints to Google Drive
Run this after each training cell so you don't lose results if Colab disconnects.

In [ ]:
import shutil, os

DRIVE_CKPT = '/content/drive/MyDrive/thesis_data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

src = '/content/mtechthesis4biaffinemultilingual-parser/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, DRIVE_CKPT, dirs_exist_ok=True)
    print('Checkpoints saved to Google Drive.')
    for root, dirs, files in os.walk(DRIVE_CKPT):
        for f in files:
            fp = os.path.join(root, f)
            print(f'  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)')
else:
    print('No checkpoints directory found yet.')

---
## Quick test (optional — fast F vs G vs H comparison without full training)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

# 2000 sentences, 10 epochs — finishes in ~15 min on T4
!python3 quick_test.py --sents 2000 --epochs 10